In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import csv
url_base = 'https://books.toscrape.com/'

#unir url base con url relativa lo planteee asi por que me daba errrores codigo sacado de php  para obtener la ruta de los libros xdd 
def unir_url(base, relativa):
    if relativa.startswith('http'):
        return relativa
    esquema, _, resto = base.partition('://')
    autoridad = resto.split('/', 1)[0]
    ruta = resto.split('/', 1)[1] if '/' in resto else ''
    directorio = ruta.rsplit('/', 1)[0] if '/' in ruta else ''
    nueva_ruta = f'{directorio}/{relativa}' if directorio else relativa
    return f'{esquema}://{autoridad}/{nueva_ruta}'

In [12]:
def extraer_libros_de_pagina(url):
    try:
        respuesta = requests.get(url)
        respuesta.raise_for_status()
    except requests.exceptions.RequestException as error:
        print(f"Error: {error}")
        return [], None

    sopa = BeautifulSoup(respuesta.text, 'html.parser')


    contenedores_libros = sopa.find_all('article', class_='product_pod')

    datos_libros = []
    for contenedor in contenedores_libros:
        enlace = contenedor.find('h3').find('a')
        titulo = enlace['title']  # Título completo desde el atributo title
        precio = contenedor.find('p', class_='price_color').text.strip()  # Precio
        disponibilidad = contenedor.find('p', class_='instock availability').text.strip()  # Disponibilidad

        datos_libros.append({
            'Título': titulo,
            'Precio': precio,
            'Disponibilidad': disponibilidad
        })

    return datos_libros, sopa

In [ ]:
def extraer_varias_paginas(url_base):
    todos_los_libros = []
    url_actual = url_base
    numero_pagina = 1

    # Recorre las páginas siguiendo el enlace next hasta que ya no haya más
    while url_actual:
        print(f"Extrayendo página {numero_pagina}: {url_actual}")
        libros, sopa = extraer_libros_de_pagina(url_actual)
        if libros:
            todos_los_libros.extend(libros)
        else:
            break

        # Paginador: busca el enlace de la siguiente página
        enlace_siguiente = sopa.find('li', class_='next')
        if enlace_siguiente and enlace_siguiente.find('a'):
            url_actual = unir_url(url_actual, enlace_siguiente.find('a')['href'])
            numero_pagina += 1
            time.sleep(5)  
        else:
            url_actual = None

    return todos_los_libros

In [14]:
def guardar_en_csv(datos, nombre_archivo):
    if not datos:  
        print("No hay datos para guardar.")
        return

    columnas = datos[0].keys() 
    with open(nombre_archivo, 'w', newline='', encoding='utf-8-sig') as archivo:
        escritor = csv.DictWriter(archivo, fieldnames=columnas)
        escritor.writeheader() 
        escritor.writerows(datos)  # Escribe todos los libros
    print(f"Datos guardados en {nombre_archivo}")

In [15]:

todos_los_libros = extraer_varias_paginas(url_base)
guardar_en_csv(todos_los_libros, 'catalogo_libros.csv')

Extrayendo página 1: https://books.toscrape.com/
Extrayendo página 2: https://books.toscrape.com/catalogue/page-2.html
Extrayendo página 3: https://books.toscrape.com/catalogue/page-3.html
Extrayendo página 4: https://books.toscrape.com/catalogue/page-4.html
Extrayendo página 5: https://books.toscrape.com/catalogue/page-5.html
Extrayendo página 6: https://books.toscrape.com/catalogue/page-6.html
Extrayendo página 7: https://books.toscrape.com/catalogue/page-7.html
Extrayendo página 8: https://books.toscrape.com/catalogue/page-8.html
Extrayendo página 9: https://books.toscrape.com/catalogue/page-9.html
Extrayendo página 10: https://books.toscrape.com/catalogue/page-10.html
Extrayendo página 11: https://books.toscrape.com/catalogue/page-11.html
Extrayendo página 12: https://books.toscrape.com/catalogue/page-12.html
Extrayendo página 13: https://books.toscrape.com/catalogue/page-13.html
Extrayendo página 14: https://books.toscrape.com/catalogue/page-14.html
Extrayendo página 15: https://b